# Stock Price Forecasting with LSTM

## Objectives
Forecast next-day **closing price** using past window of prices (univariate time series).

## Theory
LSTM remembers patterns over **sequences** of timesteps; we slide a window of `lookback` days to predict day $t+1$.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Business Context
Banks and funds use forecasts for risk; retail investors for planning (not financial advice).


In [ ]:
import yfinance as yf

ticker = "AAPL"
df = yf.download(ticker, start="2015-01-01", end="2024-12-31", progress=False)
df = df[["Close"]].dropna()
print(df.tail())


In [ ]:
sns.lineplot(data=df["Close"])
plt.title(f"{ticker} closing price")
plt.show()


In [ ]:
lookback = 60
data = df["Close"].values.astype(np.float32)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(data.reshape(-1, 1)).flatten()

def make_sequences(series, lb):
    X, y = [], []
    for i in range(lb, len(series)):
        X.append(series[i-lb:i])
        y.append(series[i])
    return np.array(X), np.array(y)

X, y = make_sequences(scaled, lookback)
X = X[..., np.newaxis]  # (samples, timesteps, features)

n = len(X)
train_end = int(n * 0.7)
val_end = int(n * 0.85)
X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]


In [ ]:
model = models.Sequential([
    layers.Input(shape=(lookback, 1)),
    layers.LSTM(50, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(50),
    layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()


In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[
        callbacks.EarlyStopping(patience=8, restore_best_weights=True),
        callbacks.ModelCheckpoint("lstm_stock_best.keras", save_best_only=True),
    ],
    verbose=1,
)


In [ ]:
pred_scaled = model.predict(X_test, verbose=0).ravel()
pred = scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
actual = scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()
print("RMSE:", np.sqrt(mean_squared_error(actual, pred)))
plt.plot(actual, label="Actual")
plt.plot(pred, label="Predicted")
plt.legend()
plt.title("Stock close price forecast (test)")
plt.show()


In [ ]:
import joblib
joblib.dump(scaler, "stock_scaler.pkl")
model.save("lstm_stock_final.keras")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
